**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/wong-campos-2023"], check=True)
    os.chdir("/content/encore/entries/wong-campos-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Wong-Campos et al. (2023) from dandiset 001791

Paper. Wong-Campos JD, Park P, Davis H, Qi Y, Fan LZ, Marblestone AH, Grimm JB, Lavis LD,
Cohen AE et al. "Voltage dynamics of dendritic integration and back-propagation in vivo".
bioRxiv preprint, [doi:10.1101/2023.05.25.542363](https://doi.org/10.1101/2023.05.25.542363).
The claims file was written from version 1 of the preprint (main text, legends, Methods and the
legends of Figs. S1 to S14, from its JATS XML); no PDF was read.

Dandiset. [DANDI:001791](https://dandiarchive.org/dandiset/001791/0.260424.0313),
"Voltage dynamics of cortical dendrites in vivo", version 0.260424.0313. The paper cites no
version (the preprint is older than the dandiset). This is the only published version, so it is
the one pinned; the draft holds the same 44 assets with the same paths and sizes (checked
below).

What the dandiset holds. The traces of the authors' pipeline grouped by figure, some
per-figure summary tables, and nine HiLo z-stacks. Every file is deposited twice, byte for byte.
The file names and contents follow a later version of the paper than the preprint the claims
were read from: they name panels the preprint does not have (Figs. 2h, 2i/2j, 3i as traces), and
the summary tables count 12 neurons for Fig. 2 (preprint: 5), 7 neurons for Fig. 3d (preprint:
3) and 238 triplets in 5 neurons for Fig. 4d (preprint: 186 in 3). There are no imaging movies,
no morphology reconstructions, and no recordings of dendrite-targeted stimulation.

| Claim | Tested here |
|---|---|
| fig1d-light-intensity-spike-rate | tested (pulse times inferred from the traces) |
| fig1fg-dendrite-peaks-after-soma | tested |
| fig2c-no-trial-variation | blocked, data missing (no Fig. 2 traces) |
| fig2e-half-amplitude-distance | tested on the deposited amplitudes and distances |
| fig2f-delay-range | tested on the deposited delays |
| fig2f-conduction-speed | tested on the deposited delays |
| fig3d-dendrite-cross-correlation | tested |
| figs9-shot-noise-bound | blocked, data missing (no pixel-level movies) |
| fig3i-coupling-vs-distance | blocked, data missing (no dendrite-targeted stimulation) |
| figs11-dendritic-spikes-rare | blocked, data missing (no dendrite-targeted stimulation) |
| fig4a-doublet-isi | tested |
| fig4bc-doublet-first-fails | tested |
| fig4d-accelerating-amplifies | tested |
| fig4fg-opto-first-bap-attenuated | selection made here; blocked, no distal dendrite |
| fig4h-biphasic-bap-amplitude | blocked, depends on fig4fg-opto-first-bap-attenuated (shown, not deciding) |
| figs13-distal-more-modulated | blocked, depends on fig4h-biphasic-bap-amplitude (shown, not deciding) |
| fig4ijk-sensory-first-fails | tested |
| fig4l-sensory-plateau | tested |
| figs12-spontaneous-failures-both-states | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

import dendvolt as dv

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
fig1 = dv.load_fig1()
fig2 = dv.load_fig2()
n3_an = dv.load_zscore_file("fig3_neuron_anesthetized")
n3_aw = dv.load_zscore_file("fig3_neuron_awake")
fig3_traces = dv.load_fig3_traces()
fig3_corr = dv.load_fig3_corr()
fig3e = dv.load_zscore_file("fig3e")
spont, spont_extra = dv.load_fig4_spontaneous()
opto = dv.load_fig4_optogenetic()
sensory = dv.load_fig4_sensory()
print(f"loaded dandiset {dv.DANDISET_ID} version {dv.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
assets = dv.assets()
print("Assets in the pinned version:", len(assets), "; total size (GB):",
      round(sum(a["size"] for a in assets.values()) / 1e9, 1))
draft = {}
url = f"{dv.API}/dandisets/{dv.DANDISET_ID}/versions/draft/assets/?page_size=200"
while url:
    page = dv._get_json(url)
    draft.update({a["path"]: a["size"] for a in page["results"]})
    url = page.get("next")
print("Draft has the same paths and sizes:", draft == {p: a["size"] for p, a in assets.items()})

digests = {}
for p in assets:
    d = dv.asset_metadata(p).get("digest", {})
    digests.setdefault(d.get("dandi:sha2-256"), []).append(p)
groups = list(digests.values())
print("Distinct SHA-256 digests:", len(groups), "; group sizes:", sorted({len(g) for g in groups}))
pairs_ok = all(sorted([k, v]) in [sorted(g) for g in groups] for k, v in dv.DUPLICATES.items())
print("Every file read here has one byte-identical twin, as listed in dendvolt.DUPLICATES:", pairs_ok)
print("\nFile kinds (one of each pair):")
rows = []
for g in groups:
    p = sorted(g, key=lambda s: s.startswith("sub-"))[0]
    rows.append({"file": p, "twin": [q for q in g if q != p][0], "MB": round(assets[p]["size"] / 1e6, 1)})
print(pd.DataFrame(rows).sort_values("MB").to_string(index=False))
meta = dv.version_metadata()
print("\nArchive summary: subjects", meta["assetsSummary"].get("numberOfSubjects"),
      "; variables", meta["assetsSummary"].get("variableMeasured"))

The 44 assets are 22 pairs of byte-identical files (same SHA-256), each pair once at the top
level under a figure name and once under a `sub-*` folder. Each recording is counted once here.
The nine files over 1 GB are HiLo z-stacks (structural images, one per neuron); they are not used.
The archive's count of 19 subjects counts pooled pseudo-subjects (`pooled_12neurons_12mice` and
so on) and duplicates under two spellings (`Barrel500-2` and `barrel500_2`).

In [ ]:
zs = [p for p in assets if p.endswith("_zstack.nwb")]
h = dv.open_h5(dv.FILES["fig1"])
print("Fig. 1 file:", fig1["session"], "| subject", fig1["subject"])
print("  ", fig1["description"])
print("   shape", fig1["data"].shape, "unit", fig1["unit"], "rate", fig1["rate"], "Hz; duration",
      fig1["data"].shape[0] / fig1["rate"], "s; range", fig1["data"].min().round(2), "to", fig1["data"].max().round(2))
print("   pixel masks:", h["processing/ophys/ImageSegmentation/PlaneSegmentation/pixel_mask"].attrs["description"])
print("   stimulus or interval tables in the file:", list(h["stimulus/presentation"].keys()), list(h["intervals"].keys()))
for name, z in (("Fig. 3 neuron, anesthetized file", n3_an), ("Fig. 3 neuron, awake file", n3_aw), ("figure3e file", fig3e)):
    print(name, "|", z["session"], "| subject", z["subject"], "|", z["description"], "| shape", z["data"].shape)
print("Anesthetized and awake files of the Fig. 3 neuron hold different traces:",
      not np.array_equal(n3_an["data"], n3_aw["data"]))
same = [r["name"] for r in fig3_traces if r["data"].shape == fig3e["data"].shape and np.allclose(r["data"], fig3e["data"])]
print("figure3e file equals figure3i recording:", same)

The Fig. 1 file holds 10 s of z-scored traces (soma and four dendrites) and no record of the
blue light pulses; its pixel masks are placeholders. Both files of the Fig. 3 neuron have the
session description "awake mouse", including the one named anesthetized at 1% isoflurane; the
file names and the two different traces are taken as the state. The figure3e file is a third
copy of figure3i recording 3 and is not counted again.

In [ ]:
print("Fig. 2 summary series (as deposited):")
for (role, name), v in fig2.items():
    print(f"  {dv.FILES[role]:42s} {name:28s} n={len(v['data']):3d} unit={v['unit']:6s} "
          f"values {np.nanmin(v['data']):.3f} to {np.nanmax(v['data']):.3f}; x {np.nanmin(v['x']):.1f} to {np.nanmax(v['x']):.1f}")
amp_all = fig2[("fig2_amplitude", "all_amplitude_vs_distance")]
big = fig2[("fig2_delay", "all_amplitude_vs_distance")]
print("\n'all' series of the delay file: first 53 rows equal the amplitude file's 'all' series:",
      np.allclose(big["data"][:53], amp_all["data"]) and np.allclose(big["x"][:53], amp_all["x"]))
print("rows 53-69, 70-86 and 87-103 are the same 17 rows repeated:",
      np.allclose(big["data"][53:70], big["data"][70:87]) and np.allclose(big["data"][53:70], big["data"][87:104]))
lr_d = fig2[("fig2_delay", "LR_amplitude_vs_distance")]
st_d = fig2[("fig2_delay", "ST_amplitude_vs_distance")]
print("Delay file LR + ST rows:", len(lr_d["data"]) + len(st_d["data"]),
      "; amplitude file LR + ST rows:", len(fig2[("fig2_amplitude", "LR_amplitude_vs_distance")]["data"]) +
      len(fig2[("fig2_amplitude", "ST_amplitude_vs_distance")]["data"]))
print("Figure 2h file states n = 69 dendrites, 12 neurons, 12 mice.")

Disagreement between the description and the values, trusted to the values: in
`figure2g_bAP_propagation_delay.nwb` the series are named `*_amplitude_vs_distance` and have the
unit -dF/F, but the LR and ST series hold values from 0 to 4.17 that are delays in ms (the file
name, and the range of the Fig. 2f delays). Its `all_amplitude_vs_distance` series is the 53
amplitudes of the amplitude file followed by 17 more amplitude rows repeated three times (104
rows), and its `branch_order_vs_amplitude` series holds delays. The amplitude file has 53 sites,
the delay file 69; with the 17 extra rows there are 70 amplitude sites. Distances are in the
timestamps field, as the descriptions say. LR is the broadly trafficked (Lucy-Rho) CheRiff
construct and ST the soma-targeted one (file notes).

In [ ]:
print("Files of the trial-to-trial bAP amplitude correlations (Figs. 2h and 2i/2j), every series:")
fig2hj = {}
for role in ("fig2_soma_dendrite_corr", "fig2_dendrite_dendrite_corr"):
    nwb = dv.open_nwb(dv.FILES[role])
    print(f"\n{dv.FILES[role]} | {nwb.session_description}")
    print("  acquisition:", list(nwb.acquisition), "| stimulus:", list(nwb.stimulus), "| notes:", nwb.notes)
    for mn, mod in nwb.processing.items():
        for name, ts in mod.data_interfaces.items():
            d, t = dv.series(ts)
            fig2hj[(role, name)] = (d, t)
            print(f"  {mn}/{name}: shape {d.shape}, unit {ts.unit}, values {d.min():.3f} to {d.max():.3f}; "
                  f"timestamps {t.min():.3f} to {t.max():.3f}\n    {ts.description}")
dvals = fig2hj[("fig2_soma_dendrite_corr", "soma_vs_dendrite_amplitudes")]
print("\nNot deciding: the 994 soma and dendrite pairs of normalized amplitudes (each divided by its STA"
      " amplitude) pool every site without a site label; dendritic s.d.", dvals[0].std(ddof=1).round(3),
      ", somatic s.d.", dvals[1].std(ddof=1).round(3), ", fraction of dendritic values below 0.5:",
      np.mean(dvals[0] < 0.5).round(3))

The two files hold, for each bAP event, the soma and dendrite amplitudes divided by their STA
amplitude, pooled over sites with no site label, and per-site soma-dendrite and
dendrite-dendrite correlations of those values. They hold no single-trial traces and no
baseline-noise values, and the amplitudes cannot be grouped by site. Neither reading of the
fig2c criterion can therefore be run (reading (a) needs the same measure in pre-stimulus windows,
reading (b) the baseline noise of each site), and fig2c-no-trial-variation stays blocked for
missing data. The pooled values above are shown, not used.

In [ ]:
print("Fig. 3 traces (figure3i file) and the Fig. 3 neuron:")
rows = [{"recording": r["name"], "state": r["state"], "mouse": r["mouse"], "ROIs": r["data"].shape[1],
         "frames": r["data"].shape[0], "s": r["data"].shape[0] / r["rate"],
         "mean": r["data"].mean().round(3), "sd": r["data"].std().round(3)} for r in fig3_traces]
rows += [{"recording": "barrel1 " + s, "state": s, "mouse": "barrel1", "ROIs": z["data"].shape[1],
          "frames": z["data"].shape[0], "s": z["data"].shape[0] / 500, "mean": z["data"].mean().round(3),
          "sd": z["data"].std().round(3)} for s, z in (("anesthetized", n3_an), ("awake", n3_aw))]
print(pd.DataFrame(rows).to_string(index=False))
print("\nDescription of a figure3i recording:", fig3_traces[5]["description"])
for k, v in fig3_corr.items():
    print(f"{k}: {len(v['corr'])} pairs, values {v['corr'].min():.3f} to {v['corr'].max():.3f}, "
          f"{int((v['corr'] > 1).sum())} above 1; distance {v['distance'].min():.0f} to {v['distance'].max():.0f} um, "
          f"{int((v['distance'] == 0).sum())} at distance 0")

The z-scored traces have mean 0 and s.d. 1, as stated. The figure3i descriptions say "Column 1
= soma (if present)", so which recordings include a soma is not stated; this matters for which
columns are dendrites. The deposited Fig. 3d values ("peak cross-correlation within +/- 2 s",
7 neurons) include 30 values above 1, which a correlation cannot take, and 61 pairs at distance 0.

In [ ]:
print("Is column 1 of each figure3i recording a soma? Spikes at 5 robust s.d. per column (Hz), the STA delay"
      " of each other column after column 1 (parabola, ms), and the 99.9th percentile of each high-pass column"
      " in robust s.d.:")
rows = []
for r in fig3_traces:
    x = r["data"]
    sp1 = dv.detect_spikes(x[:, 0], 5)
    dl = [round(dv.sta_delay_ms(x[:, 0], x[:, c], sp1, True)[0], 2) for c in range(1, x.shape[1])] if len(sp1) >= 3 else []
    hpx = dv.highpass(x)
    rows.append({"recording": r["name"], "columns": x.shape[1], "col1_spikes": len(sp1),
                 "rate_Hz_by_column": np.round([len(dv.detect_spikes(x[:, c], 5)) / (len(x) / dv.FS) for c in range(x.shape[1])], 2).tolist(),
                 "delay_after_col1_ms": dl, "n_delay_gt_0": int(np.sum(np.array(dl) > 0)) if dl else None,
                 "p99.9_in_rsd": [round(np.percentile(hpx[:, c], 99.9) / dv.robust_sd(hpx[:, c]), 1) for c in range(x.shape[1])]})
col1_check = pd.DataFrame(rows)
print(col1_check.to_string(index=False))

In the awake recordings, column 1 fires and the other columns peak after it (STA delays 0.08 to
1.68 ms, all positive but one of 13), consistent with a soma. In the anesthetized recordings
column 1 has 0 to 3 events at 5 s.d. in 10 to 20 s: delays can be measured only in recording 13,
where two of five columns peak before it, and in recording 14 two other columns fire more often
than column 1. The data do not show that column 1 is a soma in the anesthetized recordings; this
bears on fig3d (P2, where both choices are shown) and figs12 (below).

In [ ]:
rows = []
for r in spont:
    st = r["spike_times"]
    peak_lag = []
    soma = r["data"][:, 0]
    idx = np.round(st * dv.FS).astype(int)
    w, _ = dv.sta(soma, idx, 10, 10)
    shifts = np.array([int(np.argmax(soma[i - 2:i + 3])) - 2 for i in idx if 2 <= i < len(soma) - 2])
    rows.append({"rec": r["rec"], "mouse": r["mouse"], "ROIs": r["data"].shape[1], "frames": r["data"].shape[0],
                 "spikes": len(st), "sorted": bool(np.all(np.diff(st) > 0)),
                 "on_frame_grid": bool(np.allclose(st * dv.FS, np.round(st * dv.FS), atol=1e-3)),
                 "STA_peak_lag_frames": int(np.argmax(w)) - 10,
                 "shift_to_peak_-2..+2": [int((shifts == k).sum()) for k in (-2, -1, 0, 1, 2)], "soma_range": f"{soma.min():.1f} to {soma.max():.1f}"})
print("Fig. 4a-d spontaneous recordings (Delta F, not dF/F):")
print(pd.DataFrame(rows).to_string(index=False))
print("Mice:", sorted({r["mouse"] for r in spont}), "; dendrite columns:", sum(r["data"].shape[1] - 1 for r in spont))
print("soma_dendrite_distances rows:", len(spont_extra["soma_dendrite_distances"]["data"]),
      "; STA_dendrites columns:", spont_extra["STA_dendrites"]["data"].shape[1],
      "; STA_soma columns:", spont_extra["STA_soma"]["data"].shape[1])
acc0 = spont[0]["accelerating"]
print("rec01 accelerating table: all columns identical:", bool(np.allclose(acc0, acc0[:, :1])))
print(spont_extra["summary_accelerating"]["description"])
print("summary_accelerating shape:", spont_extra["summary_accelerating"]["data"].shape)

The file notes say spike times are frame indices; the values are seconds on the 2 ms frame grid
(the `unit` says seconds) and sorted. The soma's average around them peaks one frame (2 ms)
before the stated time in every recording: the spike times share the traces' clock, offset by
one frame. Where the deposited times are used below (as a variant), each is moved to the soma's
maximum within 2 frames (`dendvolt.align_to_peak`), since the claims' bAP windows start at the
somatic spike peak; the counts of each shift, from -2 to +2 frames, are printed. The file has 11 recordings
from 5 mice and 31 dendrite columns, while its own tables list 19 dendrite distances, 19 STA
dendrites and 12 STA soma sessions, so which columns are the same branch is not stated. The
per-recording accelerating and decelerating tables repeat the same value in every column (soma
and dendrites), and the "pooled" summary tables have 6 rows for a stated 238 triplets; neither is
used for a result.

In [ ]:
rows = []
for r in opto:
    x, fs = r["data"], r["rate"]
    on = np.round(r["stim_times"] * fs).astype(int)
    L = r["stim_frames"]
    gaps = np.diff(r["stim_times"])
    inside = np.zeros(len(x), bool)
    for o in on:
        inside[o:o + L] = True
    shifted = np.roll(inside, int(round(0.25 * fs)))
    soma = x[:, 0]
    rows.append({"rec": r["rec"], "rate_Hz": fs, "ROIs": x.shape[1], "frames": len(x), "s": round(len(x) / fs, 2),
                 "stimuli": len(on), "stim_frames": L, "stim_ms": round(L * 1000 / fs), "onset_gap_s": f"{gaps.min():.2f}-{gaps.max():.2f}",
                 "overlap": bool(gaps.min() * fs < L),
                 "soma_in_minus_out": round(soma[inside].mean() - soma[~inside].mean(), 2),
                 "same_shifted_250ms": round(soma[shifted].mean() - soma[~shifted].mean(), 2),
                 "deposited_delays_ms": np.round(r["delay_deposited"], 2).tolist()})
print("Fig. 4e-h optogenetic recordings (Delta F):")
print(pd.DataFrame(rows).to_string(index=False))
print("Descriptions of recordings 14 and 15:", [r["description"] for r in opto[13:15]])
alld = np.concatenate([r["delay_deposited"] for r in opto])
print("Deposited delays: n =", len(alld), "dendrites; range", alld.min(), "to", alld.max(), "ms; above 3 ms:", int((alld > 3).sum()))

Recordings 14 and 15 are sampled at 1,000 Hz (their `rate` and descriptions), the others at
500 Hz; every measure on these recordings below uses each recording's own rate. At that rate
both are 10 s long, their stated onsets 0.5 s apart fall where the soma steps up, and their
stimulus of 259 frames lasts 259 ms, so the windows do not overlap. (The first version of this
notebook read them at 500 Hz, which made their onsets look inconsistent and doubled their STA
delays.) The light steps the soma's fluorescence up inside the stated stimulus windows and much
less in windows shifted by 250 ms, in every recording with a positive contrast, so the stimulus
times share the traces' clock (in recordings 14 and 15 the 250 ms shift is half the 0.5 s
period, so their shifted contrast is reversed, -1.90 and -0.77). In the 500 Hz recordings the stimulus lasts 259 frames (518 ms;
the paper says 400 ms or more), in recordings 14 and 15 259 ms. No deposited delay exceeds
2.26 ms.

In [ ]:
rows = []
for r in sensory:
    a = r["aotf"]
    on_aotf = (np.flatnonzero(np.diff(a) > 0) + 1) / dv.FS
    off_aotf = (np.flatnonzero(np.diff(a) < 0) + 1) / dv.FS
    st = r["spike_times"]
    x = r["data"]
    on = np.round(r["stim_onsets"] * dv.FS).astype(int)
    resp = np.mean([x[o:o + dv.frames(300), 1:].mean() - x[o - dv.frames(300):o, 1:].mean() for o in on])
    on_s = on + dv.frames(2000)
    on_s = on_s[on_s + dv.frames(300) < len(x)]
    resp_s = np.mean([x[o:o + dv.frames(300), 1:].mean() - x[o - dv.frames(300):o, 1:].mean() for o in on_s])
    rows.append({"rec": r["rec"], "ROIs": x.shape[1], "frames": len(x), "onsets": np.round(r["stim_onsets"], 3).tolist(),
                 "AOTF_values": sorted(np.unique(a).tolist()), "AOTF_rises": len(on_aotf),
                 "AOTF_on_minus_onset_ms": sorted(set(np.round((on_aotf[1::2] - r["stim_onsets"]) * 1000).tolist())),
                 "AOTF_duration_ms": sorted(set(np.round((off_aotf[1::2] - on_aotf[1::2]) * 1000).tolist())),
                 "spikes": len(st), "spike_times_sorted": bool(np.all(np.diff(st) > 0)),
                 "dend_response": round(resp, 2), "shifted_2s": round(resp_s, 2)})
print("Fig. 4i-l sensory recordings (Delta F):")
print(pd.DataFrame(rows).to_string(index=False))
print("Air-puff trials:", sum(len(r["stim_onsets"]) for r in sensory), "(paper: n = 13 trials)")

The sensory file holds 13 air-puff trials in 3 recordings, as the paper's n = 13. The stimulus
waveform has values 0 and 1 only, rises twice per puff (two frames, 2 ms apart), and lasts 100 ms
from the stated onset, as the Methods' 100 ms puff; the dendrites depolarize after the stated
onsets and not at times shifted by 2 s, so the stimulus record shares the traces' clock. The
deposited spike times of recordings 2 and 3 are not sorted (a format problem); they are not
used for detection here, which follows the claims' criterion.

## fig1d-light-intensity-spike-rate

> In vivo, 0.5 s pulses of blue light of increasing intensity evoke somatic spiking at
> increasing rates, with corresponding bAPs in the dendrites.

The file records no light pulses. The pulse windows are inferred from the light-evoked
depolarization (`dendvolt.infer_pulses`): a square wave with the paper's 0.5 s pulse
(Results, "0.5 s pulses of widefield blue light at increasing intensity"), with a period of
1 s read from the trace (printed below) and its phase fitted to the mean of the five traces. Spikes are detected on the soma under the two readings of
the criterion (5 and 4 robust s.d.).

In [ ]:
x1 = fig1["data"]
soma1 = x1[:, 0]
onsets1, fit1 = dv.infer_pulses(x1.mean(1))
L1 = dv.frames(500)
onsets1 = [o for o in onsets1 if o + L1 <= len(x1)]
print("Mean of the five traces in 100 ms bins (rows: seconds 0 to 9; z):")
print(np.round(x1.mean(1)[:5000].reshape(100, 50).mean(1), 1).reshape(10, 10))
print("Inferred pulses: period", fit1["period_ms"], "ms, first onset", fit1["phase_ms"], "ms, on-off contrast",
      round(fit1["contrast"], 2), "z; full pulses in the 10 s file:", len(onsets1))
from scipy import ndimage
slow = ndimage.uniform_filter1d(x1.mean(1), dv.frames(50))
print("Per-pulse contrast (mean in pulse minus the 0.5 s before, z):",
      [round(slow[o:o + L1].mean() - slow[max(0, o - L1):o].mean(), 2) for o in onsets1])
print("Contrast of the best phase at each period (the period is fixed at 1 s above):",
      {per: round(float(dv.infer_pulses(x1.mean(1), periods_ms=(per,))[1]["contrast"]), 2) for per in (800, 900, 950, 1000, 1050, 1100, 1200)})
hp1 = dv.highpass(x1)
rows = []
for k in (5, 4):
    sp = dv.detect_spikes(soma1, k)
    counts = [int(((sp >= o) & (sp < o + L1)).sum()) for o in onsets1]
    rho, p = stats.spearmanr(np.arange(len(counts)), counts)
    in_pulse = sum(counts)
    hits = []
    for d in range(1, 5):
        thr = k * dv.robust_sd(hp1[:, d])
        hits.append(np.mean([hp1[s:s + dv.frames(5) + 1, d].max() > thr for s in sp if s + 3 < len(x1)]))
    any_hit = np.mean([max(hp1[s:s + 3, d].max() / (k * dv.robust_sd(hp1[:, d])) for d in range(1, 5)) > 1
                       for s in sp if s + 3 < len(x1)])
    rows.append({"K": k, "spikes": len(sp), "outside_pulses": len(sp) - in_pulse, "per_pulse": counts,
                 "rate_first_Hz": counts[0] / 0.5, "rate_last_Hz": counts[-1] / 0.5,
                 "spearman_rho": round(rho, 3), "p": round(p, 4),
                 "dend_event_within_5ms": [round(h, 2) for h in hits], "any_dendrite": round(any_hit, 2)})
r1 = pd.DataFrame(rows)
print(r1.to_string(index=False))
# Variant: pulses as found, but the rate compared between the first and the last three pulses.
for k in (5, 4):
    c = r1[r1.K == k].per_pulse.iloc[0]
    print(f"K={k}: mean spikes in the first three pulses {np.mean(c[:3]):.2f}, last three {np.mean(c[-3:]):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
t1 = np.arange(len(x1)) / dv.FS
for i in range(5):
    ax.plot(t1, x1[:, i] + 6 * i, lw=0.5, color="k" if i == 0 else f"C{i}")
for o in onsets1:
    ax.axvspan(o / dv.FS, (o + L1) / dv.FS, color="tab:blue", alpha=0.1)
sp5 = dv.detect_spikes(soma1, 5)
ax.plot(sp5 / dv.FS, soma1[sp5] + 1, "v", ms=3, color="r")
ax.set_xlabel("time (s)")
ax.set_ylabel("z-score, offset by 6 per trace")
ax.set_title("Fig. 1 neuron: soma (black) and four dendrites; shaded, inferred pulses; red, spikes (K=5)")
plt.show()
print("Plotted range of traces (z):", x1.min().round(2), "to", x1.max().round(2), "; none outside the axes")

**Result: reproduced (resting on inferred pulse times).** Nine 0.5 s depolarizing steps recur
every 1 s from 0.52 s; their contrast grows from 0.49 to 2.32 z, as for pulses of rising
intensity, and a 1 s period fits best of those from 0.8 to 1.2 s (contrast 1.28 z against 0.08
to 0.90). Somatic spikes per pulse rise with pulse order under both readings: K = 5 gives 0 in
the first pulse and 1 in the last (Spearman rho 0.56, p = 0.11; means 1.00 in the first three
and 4.33 in the last three); K = 4 gives 0 and 8 (rho 0.86, p = 0.003). So rho is above 0 and
the last pulse exceeds the first under both, as the criterion asks. In at least one dendrite an
event follows within 5 ms after 100% (K = 5) and 98% (K = 4) of somatic spikes; per dendrite,
after 72% to 100% in dendrites 1 to 3, and after only 14% (K = 5) or 33% (K = 4) in dendrite 4.

## fig1fg-dendrite-peaks-after-soma

> Spike-triggered averages of dendritic fluorescence, triggered off somatic spike times, peak
> after the somatic spike, as expected for back-propagation.

Peaks are timed by the sample of the maximum and by a parabola through it (the two readings of
fig2f-delay-range), within +/- 10 ms of the somatic spike.

In [ ]:
rows = []
for k in (5, 4):
    sp = dv.detect_spikes(soma1, k)
    for par in (False, True):
        dl = [dv.sta_delay_ms(soma1, x1[:, d], sp, par)[0] for d in range(1, 5)]
        rows.append({"K": k, "timing": "parabola" if par else "sample", "spikes": len(sp),
                     "delay_ms_D1..D4": [round(v, 2) for v in dl], "dendrites_delay_gt_0": int(np.sum(np.array(dl) > 0))})
print(pd.DataFrame(rows).to_string(index=False))
# Variant: bootstrap over spikes of the parabola delay, K = 5.
rng = np.random.default_rng(11)
sp = dv.detect_spikes(soma1, 5)
boots = np.array([[dv.sta_delay_ms(soma1, x1[:, d], rng.choice(sp, len(sp)), True)[0] for d in range(1, 5)]
                  for _ in range(500)])
print("Bootstrap (500 resamples of the K=5 spikes), parabola delay, 2.5 and 97.5 percentiles per dendrite (ms):")
print(np.percentile(boots, [2.5, 97.5], axis=0).round(2))
# Synthetic check of the parabola: two Gaussian peaks (s.d. 1.5 ms) sampled at 500 Hz, the second
# delayed by a known amount, averaged over 50 sub-frame phases of the first.
tt = np.arange(-20, 21) * 1000 / dv.FS
for true_d in (0.2, 0.5, 1.0, 1.4, 2.0):
    est = []
    for ph in np.linspace(0, 2, 50, endpoint=False):
        ws_, wd_ = np.exp(-(tt - ph) ** 2 / (2 * 1.5 ** 2)), np.exp(-(tt - ph - true_d) ** 2 / (2 * 1.5 ** 2))
        est.append((dv.peak_time(wd_, int(np.argmax(wd_)), True) - dv.peak_time(ws_, int(np.argmax(ws_)), True)) * 1000 / dv.FS)
    print(f"synthetic delay {true_d} ms: parabola gives {np.mean(est):.2f} ms on average (range {min(est):.2f} to {max(est):.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
tt = (np.arange(-20, 21)) * 1000 / dv.FS
w, n = dv.sta(soma1, sp, 20, 20)
ax.plot(tt, w / w.max(), "k", label="soma")
for d in range(1, 5):
    wd, _ = dv.sta(x1[:, d], sp, 20, 20)
    ax.plot(tt, wd / wd.max(), label=f"D{d}")
ax.axvline(0, ls="--", color="grey")
ax.set_xlim(-20, 20)
ax.set_xlabel("time from somatic spike (ms)")
ax.set_ylabel("STA / its peak")
ax.legend(fontsize=7)
plt.show()
print("Each STA divided by its own peak, so the plotted range is at most 1; n spikes =", n)

**Result: partially reproduced (resting on sub-frame peak timing).** Timed by the sample of the
maximum, every dendritic STA peaks in the same frame as the soma (delay 0.0 ms at all four
dendrites, both K), so no delay is positive. Timed by a parabola, three of four dendrites peak
after the soma (0.03 to 0.66 ms) and dendrite 4 before it (-0.27 to -0.28 ms); the bootstrap
interval is above 0 for dendrites 2 and 3 only. Sub-frame timing is adopted, since the
dandiset's own delays (Fig. 2) are given to 0.01 ms at 2 ms frames. The delays are far below the
1.5 to 4 ms the paper reports for Fig. 2. On synthetic peaks sampled at 500 Hz, the parabola
recovers true delays of 0.2 to 2.0 ms on average over sub-frame phases (0.20 to 2.00 ms), with
errors of up to about 0.3 ms at single phases.

## fig2e-half-amplitude-distance

> In anesthetized mice, the bAP peak amplitude decays to 50% at a contour distance from the soma
> of 242 ± 30 μm (95% C.I., 30 sites, 5 neurons, 5 mice), with R2 = 0.49 for amplitude against
> distance.

The dandiset holds the per-site amplitudes (-dF/F of the STA peak) and contour distances, not
the traces, so the amplitudes are the authors'. Site sets: LR, the 30 sites of the broadly
trafficked Lucy-Rho construct, the only construct the preprint describes (Fig. 1a, "a Lucy-Rho
sequence for improved membrane trafficking"), whose count equals the paper's n = 30; all 53
sites of the amplitude file (12 neurons, both constructs); the 70 sites with the 17 extra rows of
the delay file; and ST alone. Readings (a) linear fit, (b) exponential fit, as the criterion;
(c) needs the somatic spike amplitude of each recording, which is not in the dandiset.

In [ ]:
amp_lr = fig2[("fig2_amplitude", "LR_amplitude_vs_distance")]
amp_st = fig2[("fig2_amplitude", "ST_amplitude_vs_distance")]
extra_x, extra_a = big["x"][53:70], big["data"][53:70]
sets2e = {"LR (30)": (amp_lr["x"], amp_lr["data"]),
          "all, amplitude file (53)": (amp_all["x"], amp_all["data"]),
          "all with delay-file rows (70)": (np.r_[amp_all["x"], extra_x], np.r_[amp_all["data"], extra_a]),
          "ST (23)": (amp_st["x"], amp_st["data"])}


def expo(xv, a0, lam):
    return a0 * np.exp(-xv / lam)


def half_distances(dd, aa):
    lr = stats.linregress(dd, aa)
    p, _ = optimize.curve_fit(expo, dd, aa, p0=(0.15, 200), maxfev=20000)
    r2e = 1 - np.sum((aa - expo(dd, *p)) ** 2) / np.sum((aa - aa.mean()) ** 2)
    return -lr.intercept / (2 * lr.slope), lr.rvalue ** 2, np.log(2) * p[1], r2e, lr


rng = np.random.default_rng(2)
rows = []
for name, (dd, aa) in sets2e.items():
    dlin, r2l, dexp, r2e, lr = half_distances(dd, aa)
    bl, be = [], []
    for _ in range(2000):
        i = rng.integers(0, len(dd), len(dd))
        try:
            b = half_distances(dd[i], aa[i])
            bl.append(b[0])
            be.append(b[2])
        except Exception:
            pass
    rows.append({"sites": name, "n": len(dd), "lin_half_um": round(dlin, 1),
                 "lin_CI95": np.percentile(bl, [2.5, 97.5]).round(0).tolist(), "lin_R2": round(r2l, 3),
                 "exp_half_um": round(dexp, 1), "exp_CI95": np.percentile(be, [2.5, 97.5]).round(0).tolist(),
                 "exp_R2": round(r2e, 3), "slope_p": f"{lr.pvalue:.1e}",
                 "dist_range": f"{dd.min():.0f}-{dd.max():.0f}", "amp_range": f"{aa.min():.3f}-{aa.max():.3f}"})
r2e_tab = pd.DataFrame(rows)
print(r2e_tab.to_string(index=False))
print("Expected: 242 um, within 212 to 272; R2 0.49, within 0.441 to 0.539.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(amp_lr["x"], amp_lr["data"], "o", label="LR")
ax.plot(amp_st["x"], amp_st["data"], "s", label="ST")
ax.plot(extra_x, extra_a, "^", label="extra rows of the delay file")
xx = np.linspace(0, 400, 50)
lr = stats.linregress(amp_lr["x"], amp_lr["data"])
ax.plot(xx, lr.intercept + lr.slope * xx, "C0-")
ax.set_xlabel("contour distance from soma (um)")
ax.set_ylabel("bAP amplitude (-dF/F)")
ax.legend(fontsize=7)
plt.show()
print("All points within the axes: distances 24 to 380 um, amplitudes 0.048 to 0.231.")

**Result: reproduced (resting on the site set).** On the 30 LR sites, the construct the preprint
describes and the paper's n = 30, the linear fit falls to half its value at 0 at 238.9 μm
(bootstrap 95% interval 203 to 318) with R2 = 0.491, both within tolerance (212 to 272 μm; 0.441
to 0.539). The exponential reading gives 203.6 μm (R2 0.514), outside the distance range. Over
all 53 sites of the amplitude file the linear distance is 226.4 μm but R2 is 0.631; with the 70
sites, 216.2 μm and 0.558; on ST alone, 230.9 μm and 0.682, so on the other site sets only the
distance matches. Reading (c) could not be run: the somatic amplitudes are not deposited.

## fig2f-delay-range

> Back-propagation delays range from 1.5 to 4 ms over contour distances from the soma of 50 to
> 280 μm.

The delays are the deposited ones (delay file, LR and ST series); the peak-timing readings of the
criterion cannot be run without the traces.

In [ ]:
sets2f = {"LR (32)": (lr_d["x"], lr_d["data"]), "ST (37)": (st_d["x"], st_d["data"]),
          "all (69)": (np.r_[lr_d["x"], st_d["x"]], np.r_[lr_d["data"], st_d["data"]])}
lr_amp_x = set(np.round(amp_lr["x"], 3))
shared = np.array([round(v, 3) in lr_amp_x for v in lr_d["x"]])
sets2f["LR, sites also in the amplitude file (30)"] = (lr_d["x"][shared], lr_d["data"][shared])
rows = []
for name, (dd, de) in sets2f.items():
    m = (dd >= 50) & (dd <= 280)
    fit = stats.linregress(dd[dd > 50], de[dd > 50])
    rows.append({"sites": name, "in_50_280": int(m.sum()), "min_ms": round(de[m].min(), 2), "max_ms": round(de[m].max(), 2),
                 "within_1.35_4.4": int(((de[m] >= 1.35) & (de[m] <= 4.4)).sum()),
                 "below_1.35": int((de[m] < 1.35).sum()), "above_4.4": int((de[m] > 4.4).sum()),
                 "median_ms": round(np.median(de[m]), 2),
                 "fit_at_50um": round(fit.intercept + 50 * fit.slope, 2), "fit_at_280um": round(fit.intercept + 280 * fit.slope, 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("The delays are given to 0.01 ms; a value below 0.005 ms in magnitude is 0.00 at that precision.")
for name, (dd, de) in list(sets2f.items())[:2]:
    z = np.abs(de) < 0.005
    print(f"{name}: delays exactly 0: {int((de == 0).sum())}, 0.00 at 0.01 ms precision: {int(z.sum())} "
          f"(values {[f'{v:.1e}' for v in de[z]]}, at {np.round(dd[z]).tolist()} um), negative below -0.005: "
          f"{int((de <= -0.005).sum())}, distinct values at 0.01 ms: {len(np.unique(np.round(de, 2)))} of {len(de)}; "
          f"smallest five (ms): {np.sort(de)[:5].round(3).tolist()}")
print("Expected: all delays of sites at 50-280 um within 1.35 to 4.4 ms.")

**Result: not reproduced.** Of the 32 LR sites between 50 and 280 μm, 23 have delays below 1.35 ms;
delays there range from 0.00 to 3.92 ms (median 0.96 ms). Over all 61 sites in that range, 48 are
below 1.35 ms (0.00 to 4.13 ms). The upper end matches the paper's 4 ms and the lower end does
not; the pooled linear fit runs from -0.43 ms at 50 μm to 2.18 ms at 280 μm (LR). No LR delay is exactly 0, but one (at 181 μm) is 3.6e-15 ms, which is 0.00 at the
data's 0.01 ms precision; one ST delay is exactly 0 (at 185 μm) and one is -0.20 ms. Each set has
one zero delay among values that otherwise vary (29 distinct values of 32 in LR at 0.01 ms), and
whether the zeros are measured values or placeholders cannot be told from the file. Leaving the
LR zero out, 22 of 31 LR delays are still below 1.35 ms.

## fig2f-conduction-speed

> At distances over 50 μm from the soma, the bAP conduction speed is 0.09 ± 0.02 m/s (95%
> C.I., 30 sites, 5 neurons, 5 mice), with R2 = 0.43 for delay against distance.

A linear regression of delay on distance over the sites beyond 50 μm (Fig. 2f legend); speed is
the inverse of the slope (1 μm/ms = 0.001 m/s). Reading (b), per-neuron fits, cannot be run: the
files do not say which neuron each site belongs to.

In [ ]:
rng = np.random.default_rng(3)
rows = []
for name, (dd, de) in sets2f.items():
    g = dd > 50
    fit = stats.linregress(dd[g], de[g])
    bs = []
    for _ in range(2000):
        i = rng.integers(0, g.sum(), g.sum())
        s = stats.linregress(dd[g][i], de[g][i]).slope
        bs.append(1e-3 / s if s > 0 else np.nan)
    rows.append({"sites": name, "n_gt_50um": int(g.sum()), "slope_ms_per_um": round(fit.slope, 5),
                 "speed_m_s": round(1e-3 / fit.slope, 3), "speed_CI95_bootstrap": np.nanpercentile(bs, [2.5, 97.5]).round(3).tolist(),
                 "speed_from_slope_CI": [round(1e-3 / (fit.slope + 1.96 * fit.stderr), 3), round(1e-3 / (fit.slope - 1.96 * fit.stderr), 3)],
                 "R2": round(fit.rvalue ** 2, 3), "p": f"{fit.pvalue:.1e}"})
print(pd.DataFrame(rows).to_string(index=False))
print("Expected: 0.09 m/s, within 0.07 to 0.11; R2 0.43, within 0.387 to 0.473.")
fit_lr = stats.linregress(lr_d["x"][lr_d["x"] > 50], lr_d["data"][lr_d["x"] > 50])
shifted = stats.linregress(lr_d["x"][lr_d["x"] > 50], lr_d["data"][lr_d["x"] > 50] + 1.9)
print(f"\nThe speed depends only on the slope. LR delays shifted by +1.9 ms (the offset at 50 um, fig2f-delay-range): "
      f"{1e-3 / shifted.slope:.3f} m/s, R2 {shifted.rvalue ** 2:.3f}, fit {shifted.intercept + 50 * shifted.slope:.2f} ms "
      f"at 50 um and {shifted.intercept + 280 * shifted.slope:.2f} ms at 280 um.")
print(f"Speed implied by the paper's range, 230 um over 1.5 to 4 ms: {230e-3 / 2.5:.3f} m/s; "
      f"LR fit rise from 50 to 280 um: {230 * fit_lr.slope:.2f} ms.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lr_d["x"], lr_d["data"], "o", label="LR")
ax.plot(st_d["x"], st_d["data"], "s", label="ST")
g = lr_d["x"] > 50
fit = stats.linregress(lr_d["x"][g], lr_d["data"][g])
xx = np.linspace(50, 400, 20)
ax.plot(xx, fit.intercept + fit.slope * xx, "C0-")
ax.axvspan(50, 280, color="grey", alpha=0.1)
ax.axhspan(1.5, 4, color="tab:red", alpha=0.07)
ax.set_xlabel("contour distance from soma (um)")
ax.set_ylabel("deposited delay (ms)")
ax.legend(fontsize=7)
plt.show()
print("Delays -0.20 to 4.17 ms and distances 24 to 380 um, all within the axes.")

**Result: reproduced (resting on the site set).** On the 32 LR sites beyond 50 μm the pooled fit
gives 0.088 m/s (bootstrap 95% interval 0.063 to 0.131) with R2 = 0.462, both within tolerance
(0.07 to 0.11 m/s; 0.387 to 0.473). On the 30 LR sites shared with the amplitude file, 0.085 m/s
and R2 0.427. Over all 65 sites, 0.081 m/s with R2 0.563, and on ST alone 0.075 m/s with R2
0.655: the speed is within tolerance on every set, the R2 only on LR. Per-neuron fits (reading
(b)) could not be run. The speed reproduces on the same deposited delays whose range does not
(fig2f-delay-range), on which the claims file makes it depend. The speed comes from the slope only:
the LR fit rises 2.62 ms from 50 to 280 μm, against the 2.5 ms of the paper's 1.5 to 4 ms range
(0.092 m/s), and the delays shifted up by 1.9 ms give the same speed and R2 with a fit from 1.47
to 4.08 ms. So the deposited delays differ from the paper's by an offset of about 1.9 ms and
agree with them in slope. If the offset comes from how the delays were measured (see fig4fg), the
speed does not depend on it; the range does.

## fig3d-dendrite-cross-correlation

> Spontaneous voltage in pairs of dendritic branches of the same neuron is highly
> cross-correlated, 91 ± 3% (mean ± s.d.) in the anesthetized state and 73 ± 4% in the awake
> state (23 branch pairs, 3 neurons, 3 mice).

Pair sets, since the 23 pairs cannot be identified: P1, the Fig. 3 neuron (five dendrites, both
states, the barrel1 files); P2, the three neurons recorded in both states (the Fig. 3 neuron,
Barrel500-2 and Barrel300-4), which matches the paper's 3 neurons; P3, every recording of each
state. For the figure3i recordings, column 1 is treated as the soma (left out of dendrite pairs)
or as a dendrite, since the file says "soma (if present)". Measures (a) zero-lag Pearson,
(b) peak of the normalized cross-correlation within +/- 10 ms, (c) as (a) without 10 ms around
each somatic spike (K = 5 on column 1) where a soma is taken to be present.

In [ ]:
def pair_values(x, dend_cols, measure, soma_col=None):
    excl = None
    if measure == "nospike" and soma_col is not None:
        sp = dv.detect_spikes(x[:, soma_col], 5)
        excl = np.zeros(len(x), bool)
        for s in sp:
            excl[max(0, s - dv.frames(5)):s + dv.frames(5) + 1] = True
    m = "peak" if measure == "peak" else "zero"
    return [dv.pair_correlation(x[:, i], x[:, j], m, exclude=excl)
            for a, i in enumerate(dend_cols) for j in dend_cols[a + 1:]]


recs3 = [{"neuron": "barrel1 (Fig. 3)", "state": "anesthetized", "data": n3_an["data"], "has_soma_col": False},
         {"neuron": "barrel1 (Fig. 3)", "state": "awake", "data": n3_aw["data"], "has_soma_col": False}]
recs3 += [{"neuron": r["mouse"], "state": r["state"], "data": r["data"], "has_soma_col": True, "name": r["name"]}
          for r in fig3_traces]
both = {"barrel1 (Fig. 3)", "Barrel500-2", "Barrel300-4"}
rows = []
for col1 in ("soma", "dendrite"):
    for measure in ("zero", "peak", "nospike"):
        for pset in ("P1", "P2", "P3"):
            out = {}
            for state in ("anesthetized", "awake"):
                vals, neurons = [], set()
                for r in recs3:
                    if r["state"] != state:
                        continue
                    if pset == "P1" and r["neuron"] != "barrel1 (Fig. 3)":
                        continue
                    if pset == "P2" and r["neuron"] not in both:
                        continue
                    x = r["data"]
                    soma_col = 0 if (r["has_soma_col"] and col1 == "soma") else None
                    cols = list(range(1 if soma_col == 0 else 0, x.shape[1]))
                    v = pair_values(x, cols, measure, soma_col)
                    vals += v
                    if v:
                        neurons.add(r["neuron"])
                out[state] = (np.mean(vals), np.std(vals, ddof=1), len(vals), len(neurons))
            if pset == "P1" and col1 == "dendrite":
                continue
            an, aw = out["anesthetized"], out["awake"]
            rows.append({"col1": col1, "measure": measure, "pairs": pset,
                         "anes_mean": round(an[0], 3), "anes_sd": round(an[1], 3), "anes_n": an[2], "anes_neurons": an[3],
                         "awake_mean": round(aw[0], 3), "awake_sd": round(aw[1], 3), "awake_n": aw[2], "awake_neurons": aw[3],
                         "anes_ok": abs(an[0] - 0.91) <= 0.03, "awake_ok": abs(aw[0] - 0.73) <= 0.04, "anes_gt_awake": an[0] > aw[0]})
r3d = pd.DataFrame(rows)
print(f"Reading (c) leaves out frames s-{dv.frames(5)} to s+{dv.frames(5)} around each somatic spike s, "
      f"{2 * dv.frames(5) + 1} frames ({(2 * dv.frames(5) + 1) * 1000 / dv.FS:.0f} ms).")
print(r3d.to_string(index=False))
print("Expected: anesthetized 0.91 within 0.88 to 0.94; awake 0.73 within 0.69 to 0.77; anesthetized > awake.")

In [ ]:
print("Per recording, zero-lag Pearson, mean over dendrite pairs (column 1 as soma):")
rows = []
for r in recs3:
    x = r["data"]
    cols = list(range(1, x.shape[1])) if r["has_soma_col"] else list(range(x.shape[1]))
    v = pair_values(x, cols, "zero")
    rows.append({"recording": r.get("name", r["neuron"] + " " + r["state"]), "state": r["state"],
                 "dend_pairs": len(v), "mean": round(np.mean(v), 3) if v else None})
print(pd.DataFrame(rows).to_string(index=False))
print("\nVariant: the deposited Fig. 3d values (peak within +/- 2 s, 7 neurons), pairs at distance > 0:")
for k, v in fig3_corr.items():
    m = v["distance"] > 0
    print(f"  {k}: mean {v['corr'][m].mean():.3f}, s.d. {v['corr'][m].std(ddof=1):.3f}, n = {m.sum()}")

**Result: partially reproduced (resting on the pair set).** For the Fig. 3 neuron (10 pairs of
five dendrites), the mean cross-correlation is 0.903 ± 0.013 anesthetized and 0.732 ± 0.034 awake
at zero lag, 0.904 and 0.735 at the peak within 10 ms; both are within tolerance under all three
measures, and anesthetized exceeds awake. Over the three neurons recorded in both states (the
paper's count), the means are 0.863 (14 pairs) and 0.803 (21 pairs) with column 1 as soma, or
0.821 and 0.808 with it as a dendrite: anesthetized still exceeds awake, but neither mean is
within tolerance. Over every recording, anesthetized is 0.750 and awake 0.804. The 23 pairs of
the paper cannot be identified in the dandiset. P1 is one of the paper's three neurons, so the
match there covers only part of the claim; P2 takes the two other neurons from the mouse names
of the figure3i recordings (Barrel500-2 and Barrel300-4), and the column 1 check above does not
show that column 1 of their anesthetized recordings is a soma, which is why both choices are
shown. Reading (c) leaves out 5 frames (10 ms) around each spike, as the criterion says.

## fig4a-doublet-isi

> In awake mice some cells fire spontaneous spike doublets with a narrow distribution of
> inter-spike intervals, 18 ± 7 ms (mean ± s.d., 216 events, 3 neurons, 3 mice).

Spikes detected on the soma column at 5 and 4 robust s.d. (criterion); the deposited spike times
are shown as a variant. Doublet readings (a) isolated pairs and (b) any pair below 75 ms.

In [ ]:
def spikes_of(r, src):
    if src == "deposited":
        return dv.align_to_peak(r["data"][:, 0], np.sort(np.round(r["spike_times"] * dv.FS).astype(int)))
    return dv.detect_spikes(r["data"][:, 0], 5 if src == "K5" else 4)


SOURCES = ("K5", "K4", "deposited")

# Agreement of the two detection readings with the authors' spike times deposited with these
# recordings (aligned to the soma peak), matched within +/- 1 frame.
rows = []
for r in spont:
    dep = spikes_of(r, "deposited")
    row = {"rec": r["rec"], "mouse": r["mouse"], "deposited": len(dep)}
    for src in ("K5", "K4"):
        det = spikes_of(r, src)
        hit = sum(np.any(np.abs(det - d) <= 1) for d in dep)
        row[f"{src}_detected"] = len(det)
        row[f"{src}_recall"] = round(hit / len(dep), 2)
        row[f"{src}_precision"] = round(sum(np.any(np.abs(dep - d) <= 1) for d in det) / max(len(det), 1), 2)
    rows.append(row)
agree = pd.DataFrame(rows)
print("Detection against the deposited spike times:")
print(agree.to_string(index=False))
tot = agree[["deposited", "K5_detected", "K4_detected"]].sum()
print("Totals:", tot.to_dict(), "; mean recall K5", agree.K5_recall.mean().round(2), "K4", agree.K4_recall.mean().round(2),
      "; mean precision K5", agree.K5_precision.mean().round(2), "K4", agree.K4_precision.mean().round(2))
rows, permouse = [], []
for src in SOURCES:
    for reading in ("a", "b"):
        isis = []
        for r in spont:
            sp = spikes_of(r, src)
            d = dv.doublet_starts(sp, reading)
            v = dv.isi_ms(sp)[d] if d else np.array([])
            isis.append(v)
            permouse.append({"spikes": src, "reading": reading, "mouse": r["mouse"], "n": len(v), "isi": v})
        v = np.concatenate(isis)
        rows.append({"spikes": src, "reading": reading, "doublets": len(v), "mean_ms": round(v.mean(), 1),
                     "sd_ms": round(v.std(ddof=1), 1), "median_ms": round(np.median(v), 1),
                     "mean_ok": abs(v.mean() - 18) <= 1.8, "sd_ok": abs(v.std(ddof=1) - 7) <= 0.7,
                     "count_ok": abs(len(v) - 216) <= 21.6})
print(pd.DataFrame(rows).to_string(index=False))
print("Expected: mean 18 +/- 1.8 ms, s.d. 7 +/- 0.7 ms, 216 +/- 21.6 events.")
pm = pd.DataFrame(permouse)
pm = pm.groupby(["spikes", "reading", "mouse"]).apply(lambda g: pd.Series({
    "n": int(sum(g.n)), "mean_ms": np.round(np.mean(np.concatenate(list(g.isi))), 1) if sum(g.n) else np.nan,
    "sd_ms": np.round(np.std(np.concatenate(list(g.isi)), ddof=1), 1) if sum(g.n) > 1 else np.nan}))
print("\nBy mouse:")
print(pm.unstack("mouse").to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharex=True)
for a, src in zip(ax, SOURCES):
    v = np.concatenate([dv.isi_ms(spikes_of(r, src))[dv.doublet_starts(spikes_of(r, src), "a")] for r in spont])
    a.hist(v, bins=np.arange(0, 78, 4))
    a.set_title(f"{src}, reading (a), n={len(v)}")
    a.set_xlabel("doublet ISI (ms)")
plt.tight_layout()
plt.show()
print("All doublet ISIs are below 75 ms by definition and within the axes.")

**Result: not reproduced.** No reading gives the paper's doublets. Isolated doublets number 36
(K = 5, 36.6 ± 20.4 ms), 107 (K = 4, 28.1 ± 15.6 ms) and 144 with the deposited spike times
(25.2 ± 13.7 ms); with reading (b), 55, 210 and 427. Only K = 4, reading (b), gives a count
within tolerance (210), and its ISIs are 30.5 ± 17.3 ms against 18 ± 7 ms. By mouse, the
closest is mouse 5 with the deposited times, reading (b): 18.5 ± 12.8 ms over 109 doublets, a
mean within tolerance and an s.d. that is not; no mouse has an s.d. within 6.3 to 7.7 ms.

## fig4bc-doublet-first-fails

> In spontaneous spike doublets, the first spike typically fails to propagate to distal
> dendrites while the second succeeds (doublet ISIs 18 ± 7 ms, 10 branches, 3 neurons, 3 mice).

Per branch (a dendrite column of a recording), the mean over doublets of the first and second
bAP amplitudes (peak 0 to 5 or 0 to 8 ms after each somatic spike minus the mean of the 5 ms
before the first spike), and their ratio. Distal is (a) all branches or (b) branches whose STA
delay (parabola, K = 5 spikes of the recording) exceeds 3 ms. The criterion measures both bAPs
from the baseline before the first; a variant measures the second from the 5 ms before it.

In [ ]:
branch_delay = {}
for r in spont:
    sp = spikes_of(r, "K5")
    for c in range(1, r["data"].shape[1]):
        branch_delay[(r["rec"], c)] = dv.sta_delay_ms(r["data"][:, 0], r["data"][:, c], sp, True)[0]
bd = np.array(list(branch_delay.values()))
print("Branch STA delays (ms): range", np.nanmin(bd).round(2), "to", np.nanmax(bd).round(2),
      "; above 3 ms:", [k for k, v in branch_delay.items() if v > 3])

rows = []
detail = []
for src in SOURCES:
    for reading in ("a", "b"):
        for win in (5, 8):
            for base in ("first", "own"):
                ratios = {}
                for r in spont:
                    x = r["data"]
                    sp = spikes_of(r, src)
                    D = dv.doublet_starts(sp, reading)
                    if not D:
                        continue
                    for c in range(1, x.shape[1]):
                        a1 = np.nanmean([dv.bap_amplitude(x[:, c], sp[i], sp[i], win) for i in D])
                        a2 = np.nanmean([dv.bap_amplitude(x[:, c], sp[i + 1], sp[i] if base == "first" else sp[i + 1], win) for i in D])
                        ratios[(r["rec"], c)] = (a1, a2, a2 / a1, r["mouse"])
                for distal in ("all", "delay>3ms"):
                    keys = [k for k in ratios if distal == "all" or branch_delay[k] > 3]
                    rr = np.array([ratios[k][2] for k in keys])
                    rows.append({"spikes": src, "doublet": reading, "win_ms": win, "baseline": base, "distal": distal,
                                 "branches": len(keys), "frac_second_gt_first": round(np.mean(rr > 1), 2) if len(rr) else np.nan,
                                 "median_ratio": round(np.median(rr), 2) if len(rr) else np.nan,
                                 "p_signed_rank_log_ratio": dv.wilcoxon(np.log(rr[rr > 0])) if len(rr) > 1 else np.nan,
                                 "passes": bool(len(rr) and np.mean(rr > 1) >= 0.7 and np.median(rr) > 1)})
                if src == "K5" and reading == "a" and win == 5 and base == "first":
                    detail = ratios
r4bc = pd.DataFrame(rows)
print(r4bc[r4bc.baseline == "first"].drop(columns="baseline").to_string(index=False))
print("\nVariant, second bAP measured from its own 5 ms baseline:")
print(r4bc[r4bc.baseline == "own"].drop(columns="baseline").to_string(index=False))
print("\nPer branch, K5, doublet (a), 0-5 ms (first amplitude, second amplitude, ratio, mouse):")
print(pd.DataFrame([{"branch": f"{k[0]}:D{k[1]}", "first": round(v[0], 2), "second": round(v[1], 2),
                     "ratio": round(v[2], 2), "mouse": v[3], "delay_ms": round(branch_delay[k], 2)} for k, v in detail.items()]).to_string(index=False))

**Result: partially reproduced.** The criterion asks for every reading. With K = 4 and with the
deposited spike times, every reading passes: the second bAP exceeds the first in 97% (isolated
doublets) or 100% (any doublet) of the 31 branches, with median ratios of 1.34 to 1.59, in both
windows. With K = 5, isolated doublets pass (81% and 77%) but any doublet gives 68% in both
windows, below the 70% bound, so one reading fails. The distal reading (STA delay above 3 ms)
rests on 2 branches (recording 4 dendrite 2, 3.58 ms, and recording 6 dendrite 2, 3.26 ms) and
passes under every reading (ratios 1.58 to 1.95). Measuring the second bAP from its own baseline
lowers the median ratios to 1.07 to 1.15 (K = 4; 84% to 90% of branches), so part of the effect
in the criterion's measure is the first bAP's tail. The second bAP is larger, but the first is
not absent.

## fig4d-accelerating-amplifies

> In spontaneous spike trios, the third bAP is amplified relative to the second when the spike
> rate accelerates (ISI n-1 > 75 ms, ISI n < 75 ms) and suppressed when it decelerates (186 spike
> triplets, 10 branches, 3 neurons, 3 mice), so spontaneous bAP propagation is favored by an
> accelerating spike rate.

Per branch, every spike's bAP amplitude (peak 0 to 5 or 0 to 8 ms after it minus the mean of the
5 ms before it) divided by the branch's mean (Methods, "normalized bAPs by their mean
amplitude"). Readings (a), (b), (c) of the criterion, by Wilcoxon signed-rank test. A variant uses
the deposited per-spike amplitudes (`rec*_spike_amplitudes`) with the deposited spike times.
Reading (a) pools trio and branch pairs, as the criterion words it; since each trio counts once
per branch, a variant tests the per-branch median over the 31 branches. A second variant
measures the second and third bAPs from the 5 ms before the first spike of the trio, as fig4bc
measures doublets.

In [ ]:
def trio_tests(src, win, deposited_amps=False, base="own"):
    """base 'own': each bAP from the 5 ms before its own spike; 'trio': the second and third bAP from
    the 5 ms before the first spike of the trio, as fig4bc measures doublets. Amplitudes are divided
    by the branch's mean bAP amplitude (own baseline)."""
    pa, pdc, ra, rd, ba, bd, n_acc, n_dec = [], [], [], [], [], [], 0, 0
    for r in spont:
        x = r["data"]
        sp = spikes_of(r, src)
        acc, dec = dv.trios(sp)
        n_acc += len(acc)
        n_dec += len(dec)
        for c in range(1, x.shape[1]):
            if deposited_amps:
                amp = r["spike_amplitudes"][:, c].astype(float)
            else:
                amp = np.array([dv.bap_amplitude(x[:, c], s, s, win) for s in sp])
            m = np.nanmean(amp)
            if base == "trio":
                two = lambda n: (dv.bap_amplitude(x[:, c], sp[n - 1], sp[n - 2], win) / m,
                                 dv.bap_amplitude(x[:, c], sp[n], sp[n - 2], win) / m)
            else:
                two = lambda n: (amp[n - 1] / m, amp[n] / m)
            A, D = [two(n) for n in acc], [two(n) for n in dec]
            pa += A
            pdc += D
            if acc and dec:
                ra.append(np.nanmean([t / s for s, t in A]))
                rd.append(np.nanmean([t / s for s, t in D]))
                ba.append(np.nanmedian([t - s for s, t in A]))
                bd.append(np.nanmedian([t - s for s, t in D]))
    pa, pdc, ra, rd, ba, bd = map(np.array, (pa, pdc, ra, rd, ba, bd))
    out = {"acc_trios": n_acc, "dec_trios": n_dec, "pairs_acc": len(pa), "pairs_dec": len(pdc),
           "a_acc_med_3m2": np.nanmedian(pa[:, 1] - pa[:, 0]), "a_acc_p": dv.wilcoxon(pa[:, 1], pa[:, 0]),
           "a_dec_med_3m2": np.nanmedian(pdc[:, 1] - pdc[:, 0]), "a_dec_p": dv.wilcoxon(pdc[:, 1], pdc[:, 0]),
           "b_branches": len(ra), "b_med_acc_minus_dec": np.nanmedian(ra - rd), "b_p": dv.wilcoxon(ra, rd),
           "c_acc_med3": np.nanmedian(pa[:, 1]), "c_acc_p": dv.wilcoxon(pa[:, 1] - 1),
           "c_dec_med3": np.nanmedian(pdc[:, 1]), "c_dec_p": dv.wilcoxon(pdc[:, 1] - 1),
           "a_by_branch_acc_p": dv.wilcoxon(ba), "a_by_branch_dec_p": dv.wilcoxon(bd),
           "a_by_branch_acc_med": np.nanmedian(ba), "a_by_branch_dec_med": np.nanmedian(bd)}
    out["a_pass"] = out["a_acc_med_3m2"] > 0 and out["a_acc_p"] < 0.05 and out["a_dec_med_3m2"] < 0 and out["a_dec_p"] < 0.05
    out["b_pass"] = out["b_med_acc_minus_dec"] > 0 and out["b_p"] < 0.05
    out["c_pass"] = out["c_acc_med3"] > 1 and out["c_acc_p"] < 0.05 and out["c_dec_med3"] < 1 and out["c_dec_p"] < 0.05
    out["a_by_branch_pass"] = (out["a_by_branch_acc_med"] > 0 and out["a_by_branch_acc_p"] < 0.05
                               and out["a_by_branch_dec_med"] < 0 and out["a_by_branch_dec_p"] < 0.05)
    return out


rows = []
for src in SOURCES:
    for win in (5, 8):
        rows.append({"spikes": src, "win_ms": win, "base": "own", **trio_tests(src, win)})
rows.append({"spikes": "deposited", "win_ms": "deposited amplitudes", "base": "own", **trio_tests("deposited", 5, True)})
r4d = pd.DataFrame(rows)
fmt = {c: (lambda v: f"{v:.2g}") for c in r4d.columns if c.endswith("_p")}
main_cols = [c for c in r4d.columns if not c.startswith("a_by_branch")]
print(r4d[main_cols].to_string(index=False, formatters=fmt, float_format=lambda v: f"{v:.3f}"))
print("Paper: 186 triplets in 3 neurons; the dataset's summary table states 238 triplets in 5 neurons.")
print("\nVariant, reading (a) per branch (median third minus second per branch, tested over the 31 branches):")
print(r4d[["spikes", "win_ms", "b_branches", "a_by_branch_acc_med", "a_by_branch_acc_p", "a_by_branch_dec_med",
           "a_by_branch_dec_p", "a_by_branch_pass"]].to_string(index=False, formatters=fmt, float_format=lambda v: f"{v:.3f}"))
rows = [{"spikes": src, "win_ms": win, "base": "trio", **trio_tests(src, win, base="trio")} for src in SOURCES for win in (5, 8)]
r4d_trio = pd.DataFrame(rows)
print("\nVariant, second and third bAP measured from the 5 ms before the first spike of the trio:")
print(r4d_trio[main_cols].to_string(index=False, formatters=fmt, float_format=lambda v: f"{v:.3f}"))

In [ ]:
print("By mouse, K = 4, 0-5 ms, reading (a): median third minus second (n pairs)")
for m in sorted({r["mouse"] for r in spont}):
    keep = [r for r in spont if r["mouse"] == m]
    saved = spont[:]
    spont[:] = keep
    o = trio_tests("K4", 5)
    spont[:] = saved
    print(f"  mouse {m}: accelerating {o['a_acc_med_3m2']:+.3f} (n={o['pairs_acc']}, p={o['a_acc_p']:.2g}); "
          f"decelerating {o['a_dec_med_3m2']:+.3f} (n={o['pairs_dec']}, p={o['a_dec_p']:.2g})")

**Result: partially reproduced.** The criterion asks for every reading. With K = 4 (144
accelerating and 140 decelerating trios), the third bAP exceeds the second in accelerating trios
(median +0.140, p = 8.5e-16) and falls below it in decelerating ones (-0.124, p = 7e-12); the
per-branch contrast (b) is +0.277 (p = 0.0062, 31 branches); and the third bAP is above 1 when
accelerating and below 1 when decelerating, 1.083 and 0.953 in the 0-5 ms window (p = 6e-14 and
1.2e-05) and 1.076 and 0.949 in the 0-8 ms window (p = 1.1e-12 and 5.9e-05). The
deposited spike times (238 accelerating trios, the dandiset's own count) and the deposited
amplitudes pass every reading too. With K = 5 (44 and 43 trios) accelerating amplification holds
(p = 8.8e-05) and reading (b) passes, but decelerating suppression does not reach p < 0.05
((a) -0.043, p = 0.097; (c) 1.027 and 1.007, above 1), so readings (a) and (c) fail. Trios found
number 87 (K = 5), 284 (K = 4) and 473 (deposited times), against the paper's 186. Tested per
branch, reading (a) gives the same split: K = 4 and the deposited times pass, K = 5 does not
(decelerating p = 0.22 and 0.25). Measured from the baseline before the first spike of the trio,
reading (a) still passes with K = 4 and the deposited times, but (b) and (c) fail with K = 4
(b, p = 0.05 and 0.053; c, decelerating 0.966 and 0.978, p = 0.24 and 0.37), so the result also
depends on the baseline. By mouse (K = 4), the third bAP of accelerating trios is larger in all
five (+0.11 to +0.27) and that of decelerating trios smaller in four of five.

## fig4fg-opto-first-bap-attenuated

> With sustained (400 ms) blue light on the soma, the first bAP in each evoked spike train is
> highly attenuated in the dendrites relative to the soma, and subsequent bAPs propagate.

The criterion tests electrotonically distal dendrites, those with a conduction delay above 3 ms
(Methods). The selection is made first: delays as deposited (`rec*_delay`) and from the STA of
each dendrite against the soma (parabola), with spikes inside the stimulus windows detected at
K = 5 and K = 4, each recording at its own sampling rate. The test is then run on the selection,
under readings (a) (dendritic amplitude) and (b) (dendrite/soma ratio), and, not deciding, on
dendrites with lower deposited delays and on all dendrites.

In [ ]:
rows = []
for r in opto:
    x, fs = r["data"], r["rate"]
    on = np.round(r["stim_times"] * fs).astype(int)
    L = r["stim_frames"]
    for k in (5, 4):
        sp = dv.detect_spikes(x[:, 0], k, fs=fs)
        ins = np.array([s for s in sp if any(o <= s < o + L for o in on)], int)
        for c in range(1, x.shape[1]):
            d, n = dv.sta_delay_ms(x[:, 0], x[:, c], ins, True, fs=fs) if len(ins) else (np.nan, 0)
            rows.append({"rec": r["rec"], "rate_Hz": int(fs), "dend": c, "K": k, "spikes_in_stim": len(ins), "sta_delay_ms": round(d, 2),
                         "deposited_delay_ms": round(float(r["delay_deposited"][c - 1]), 2)})
sel = pd.DataFrame(rows)
print(sel.pivot_table(index=["rec", "rate_Hz", "dend", "deposited_delay_ms"], columns="K", values=["spikes_in_stim", "sta_delay_ms"]).to_string())
for k in (5, 4):
    s = sel[sel.K == k]
    print(f"K={k}: dendrites with STA delay > 3 ms: {int((s.sta_delay_ms > 3).sum())} of {len(s)}; "
          f"STA delay range {s.sta_delay_ms.min()} to {s.sta_delay_ms.max()} ms")
print("Deposited delays above 3 ms:", int((sel[sel.K == 5].deposited_delay_ms > 3).sum()), "of", len(sel[sel.K == 5]))
s4 = sel[sel.K == 4]
print("Spearman correlation of the STA (K = 4) and deposited delays over dendrites:",
      round(stats.spearmanr(s4.sta_delay_ms, s4.deposited_delay_ms, nan_policy="omit")[0], 2))
print("Trains per recording (stated onsets):", {r["rec"]: len(r["stim_times"]) for r in opto},
      "; no recording has the 10 trials of the Fig. 4g neuron.")

In [ ]:
def opto_events(k, win):
    """One row per somatic spike of a train and dendrite: order in the train, time after onset,
    dendritic bAP amplitude and somatic spike amplitude (fig4bc windows, from the 5 ms before the
    first spike of the train), each recording at its own rate. norm_a is reading (a), the
    dendritic amplitude over its 200 to 400 ms mean; norm_b reading (b), the dendrite/soma ratio
    over its 200 to 400 ms mean (dendrites whose mean is not positive are left out)."""
    out = []
    for r in opto:
        x, fs = r["data"], r["rate"]
        on = np.round(r["stim_times"] * fs).astype(int)
        L = r["stim_frames"]
        sp = dv.detect_spikes(x[:, 0], k, fs=fs)
        for t, o in enumerate(on):
            tr = sp[(sp >= o) & (sp < o + L)]
            for j, s in enumerate(tr):
                soma_amp = dv.bap_amplitude(x[:, 0], s, tr[0], win, fs=fs)
                for c in range(1, x.shape[1]):
                    out.append({"rec": r["rec"], "dend": c, "train": t, "order": j, "t_ms": (s - o) * 1000 / fs,
                                "dend_amp": dv.bap_amplitude(x[:, c], s, tr[0], win, fs=fs), "soma_amp": soma_amp})
    ev = pd.DataFrame(out)
    ev["ratio"] = ev.dend_amp / ev.soma_amp
    for col, name in (("dend_amp", "norm_a"), ("ratio", "norm_b")):
        late = ev[(ev.t_ms >= 200) & (ev.t_ms < 400)].groupby(["rec", "dend"])[col].mean()
        ref = ev.set_index(["rec", "dend"]).index.map(late).values.astype(float)
        ev[name] = np.where(ref > 0, ev[col] / ref, np.nan)
    return ev


EV = {(k, w): opto_events(k, w) for k in (5, 4) for w in (5, 8)}
delay_dep = {(r["rec"], c): float(r["delay_deposited"][c - 1]) for r in opto for c in range(1, r["data"].shape[1])}
sta4 = sel[sel.K == 4].set_index(["rec", "dend"]).sta_delay_ms
sets4fg = {"criterion: deposited delay > 3 ms": lambda rd: delay_dep[rd] > 3,
           "criterion: STA delay (K=4) > 3 ms": lambda rd: sta4.get(rd, np.nan) > 3,
           "variant: deposited delay > 1.5 ms": lambda rd: delay_dep[rd] > 1.5,
           "variant: deposited delay > 1 ms": lambda rd: delay_dep[rd] > 1,
           "variant: all dendrites": lambda rd: True}
rows = []
for name, keep in sets4fg.items():
    for (k, w), ev in EV.items():
        e = ev[[bool(keep(rd)) for rd in zip(ev.rec, ev.dend)]]
        row = {"dendrites": name, "K": k, "win_ms": w, "n_dend": e[["rec", "dend"]].drop_duplicates().shape[0]}
        ok = row["n_dend"] > 0
        for rd_ in ("a", "b"):
            f = e[e.order == 0][f"norm_{rd_}"].dropna()
            l = e[(e.order > 0) & (e.t_ms < 150)][f"norm_{rd_}"].dropna()
            mf = np.median(f) if len(f) else np.nan
            ml = np.median(l) if len(l) else np.nan
            row.update({f"{rd_}_first_n": len(f), f"{rd_}_first_med": round(mf, 2), f"{rd_}_later_n": len(l), f"{rd_}_later_med": round(ml, 2)})
            ok = ok and len(f) > 0 and len(l) > 0 and mf < ml and mf < 1
        row["passes"] = bool(ok)
        rows.append(row)
print("First bAP of each train against later bAPs in the first 150 ms, normalized by the 200-400 ms mean;"
      " (a) dendritic amplitude, (b) dendrite/soma ratio. Only the criterion rows decide; the others are variants.")
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (other: no distal dendrite).** With each recording read at its own rate
(recordings 14 and 15 at 1,000 Hz), none of the 46 dendrites has a conduction delay above 3 ms,
the criterion's rule for distal dendrites: the deposited delays reach 2.26 ms and the STA delays
1.86 ms (K = 5) or 1.95 ms (K = 4), and the two agree over dendrites (Spearman 0.88). The two STA
delays above 3 ms of the first version came from reading recording 14 at 500 Hz. The Fig. 4g
neuron (10 trials) cannot be identified, as no recording has 10 trains (9, 18 or 20). Not
deciding: on dendrites with a deposited delay above 1.5 ms (4 dendrites) or 1 ms (13), or on all
dendrites (41 to 46), the first bAP is below 1 and below the later bAPs of the first 150 ms under
both readings, both thresholds and both windows (reading (a) medians 0.59 to 0.85 against 0.66
to 1.03; reading (b) 0.85 to 0.99 against 1.01 to 1.33). The deposited Fig. 2 delays run about
1.9 ms below the paper's at 50 μm (fit -0.43 ms against 1.5 ms, fig2f) and 1.8 ms below at 280
μm, so the deposited delays may not be measured as the paper's are, and the empty selection may
come from that difference. The outcome stays with the criterion's rule.

## fig4h-biphasic-bap-amplitude and figs13-distal-more-modulated

> In distal dendrites, bAP amplitude during sustained optogenetic stimulation is biphasic in time
> after stimulus onset, suppressed for the first bAPs, enhanced in a window from 50 to 150 ms,
> then decreasing (16 trials, 7 neurons, 7 mice).

> The biphasic modulation of bAP amplitude is larger in dendrites further from the soma, by mean
> propagation delay, than in more proximal dendrites (25 branches, 7 cells, 7 mice).

Both depend on fig4fg-opto-first-bap-attenuated, and both use the same split at 3 ms, which
leaves no distal dendrite. Not deciding: the time-binned amplitudes (reading (a) of fig4fg, K = 5
and 4, 0 to 5 ms window) under the three bin readings, per dendrite, on the criterion's selection
and on the variant selections, tested over dendrites (the 16 trials of the paper cannot be
identified). For figs13 the split is also made at 1.5 ms and at the median deposited delay.

In [ ]:
BINS = {"a": [0, 50, 150, 200, 400], "b": list(range(0, 425, 25)), "c": list(range(0, 450, 50))}


def binned(ev, edges):
    """Per dendrite, the mean normalized amplitude (reading (a) of fig4fg) in each time bin."""
    e = ev.dropna(subset=["norm_a"]).copy()
    e["bin"] = pd.cut(e.t_ms, edges, right=False, labels=False)
    e = e.dropna(subset=["bin"]).astype({"bin": int})
    return e.groupby(["rec", "dend", "bin"]).norm_a.mean().unstack("bin").reindex(columns=range(len(edges) - 1))


def biphasic(tab, edges):
    lo = [i for i in range(len(edges) - 1) if edges[i] >= 50 and edges[i + 1] <= 150]
    after = [i for i in range(len(edges) - 1) if edges[i] >= 150]
    first = tab[0].dropna()
    pk = int(tab[lo].mean().idxmax())
    peak = tab[pk].dropna()
    after_mean = np.nanmean(tab[after].values)
    ok = (first.median() < 1 and dv.wilcoxon(first - 1) < 0.05 and peak.median() > 1
          and dv.wilcoxon(peak - 1) < 0.05 and after_mean < peak.mean())
    return {"n_dend": len(tab), "first_bin_med": round(first.median(), 2), "first_p": dv.wilcoxon(first - 1),
            "peak_bin_ms": f"{edges[pk]}-{edges[pk + 1]}", "peak_med": round(peak.median(), 2), "peak_p": dv.wilcoxon(peak - 1),
            "after_150_mean": round(after_mean, 2), "passes": bool(ok)}


rows = []
for name, thr in (("criterion: delay > 3 ms", 3), ("variant: delay > 1.5 ms", 1.5), ("variant: delay > 1 ms", 1),
                  ("variant: all dendrites", -np.inf)):
    for k in (5, 4):
        ev = EV[(k, 5)]
        ev = ev[[delay_dep[rd] > thr for rd in zip(ev.rec, ev.dend)]]
        for rd_, edges in BINS.items():
            if len(ev) == 0:
                rows.append({"dendrites": name, "K": k, "bins": rd_, "n_dend": 0})
                continue
            rows.append({"dendrites": name, "K": k, "bins": rd_, **biphasic(binned(ev, edges), edges)})
fmt = {c: (lambda v: f"{v:.2g}") for c in ("first_p", "peak_p")}
print("fig4h, deposited delays; first bin and peak 50-150 ms bin against 1 (signed-rank over dendrites):")
print(pd.DataFrame(rows).to_string(index=False, formatters=fmt))
rows = []
for name, split in (("criterion: 3 ms", 3.0), ("variant: 1.5 ms", 1.5), ("variant: median delay", float(np.median(list(delay_dep.values()))))):
    for k in (5, 4):
        for rd_, edges in BINS.items():
            tab = binned(EV[(k, 5)], edges)
            lo = [i for i in range(len(edges) - 1) if edges[i] >= 50 and edges[i + 1] <= 150]
            mod_a = tab[lo].max(axis=1) - tab[0]
            mod_b = tab.max(axis=1) - tab.min(axis=1)
            dist = np.array([delay_dep[i] > split for i in tab.index])
            rows.append({"split": name, "split_ms": round(split, 2), "K": k, "bins": rd_, "distal": int(dist.sum()),
                         "proximal": int((~dist).sum()),
                         "a_distal": round(mod_a[dist].mean(), 2) if dist.any() else np.nan, "a_proximal": round(mod_a[~dist].mean(), 2),
                         "b_distal": round(mod_b[dist].mean(), 2) if dist.any() else np.nan, "b_proximal": round(mod_b[~dist].mean(), 2)})
print("\nfigs13, mean modulation of distal and proximal dendrites; (a) peak 50-150 ms bin minus first bin, (b) range of bins:")
print(pd.DataFrame(rows).to_string(index=False))

**Result: both blocked (other), as they depend on fig4fg-opto-first-bap-attenuated and their
split at 3 ms leaves no distal dendrite.** Not deciding, fig4h: on no selection does the binned
amplitude pass. On all dendrites at K = 5 the first bin is below 1 (medians 0.62 to 0.85,
p = 0.0083 to 0.042), but on no selection is a 50 to 150 ms bin above 1 at p < 0.05 (peak
medians 0.40 to 1.30, p of 0.11 or more where it can be computed). Not deciding, figs13: with
the split at 1.5 ms or at the median delay (0.52 ms), distal modulation exceeds proximal in some
readings and not in others (modulation (a) at 1.5 ms: K = 5, bins (a), -0.17 against 0.14; K = 4, bins (c), 0.22
against 0.07).

## fig4ijk-sensory-first-fails

> Contralateral whisker air puffs in awake mice evoke bursts of spikes in which all detected
> dendritic spikes start as bAPs from the soma, and the first spike in the burst typically fails
> to propagate to distal dendrites while subsequent spikes succeed (13 trials).

The burst is the somatic spikes from puff onset to 300 ms after it (a window the paper does not
give; 200 and 500 ms are shown as variants). Per trial and dendrite, the first bAP amplitude and
the mean of the later ones (fig4bc windows, both from the baseline before the first spike).
Distal is read as all dendrites, and as dendrites with STA delay above 3 ms. The dendritic
spikes without a somatic spike before them are listed with the nearest somatic spike and the
soma's largest value around them.

In [ ]:
for r in sensory:
    for k in (5, 4):
        sp = dv.detect_spikes(r["data"][:, 0], k)
        print(r["rec"], f"K={k}", "STA delays (ms):",
              [round(dv.sta_delay_ms(r["data"][:, 0], r["data"][:, c], sp, True)[0], 2) for c in (1, 2)])
rows = []
for k in (5, 4):
    for win in (5, 8):
        for burst_ms in (300, 200, 500):
            per_trial, orphans, dsp = [], 0, 0
            for r in sensory:
                x = r["data"]
                sp = dv.detect_spikes(x[:, 0], k)
                for o in np.round(r["stim_onsets"] * dv.FS).astype(int):
                    b = sp[(sp >= o) & (sp < o + dv.frames(burst_ms))]
                    for c in (1, 2):
                        if len(b) >= 2:
                            a = [dv.bap_amplitude(x[:, c], s, b[0], win) for s in b]
                            per_trial.append((r["rec"], o, c, a[0], np.nanmean(a[1:])))
                        ds = dv.detect_spikes(x[:, c], k)
                        ds = ds[(ds >= o) & (ds < o + dv.frames(burst_ms))]
                        dsp += len(ds)
                        orphans += int(sum(not np.any((sp <= d) & (sp >= d - dv.frames(win))) for d in ds))
            pt = pd.DataFrame(per_trial, columns=["rec", "onset", "dend", "first", "later"])
            trials_with_burst = pt[["rec", "onset"]].drop_duplicates().shape[0]
            by_trial = pt.groupby(["rec", "onset"]).apply(lambda g: bool((g["first"] < g["later"]).all()))
            rows.append({"K": k, "win_ms": win, "burst_ms": burst_ms, "trials_with_>=2_spikes": trials_with_burst,
                         "trials_first<later_D1": int(((pt.dend == 1) & (pt["first"] < pt["later"])).sum()),
                         "trials_first<later_D2": int(((pt.dend == 2) & (pt["first"] < pt["later"])).sum()),
                         "trials_first<later_both_dend": int(by_trial.sum()),
                         "trial_dend_first<later": f"{int((pt['first'] < pt['later']).sum())}/{len(pt)}",
                         "avg_first": round(pt["first"].mean(), 2), "avg_later": round(pt["later"].mean(), 2),
                         "dend_spikes": dsp, "without_soma_spike_before": orphans})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
print("Dendritic spikes without a somatic spike in the 0-5 ms before them (300 ms bursts): the nearest somatic"
      " spike (frames, + = after the dendritic spike) and the largest high-pass soma value from 3 frames before"
      " to 2 frames after, as a fraction of the somatic threshold:")
rows = []
for k in (5, 4):
    for r in sensory:
        x = r["data"]
        sp = dv.detect_spikes(x[:, 0], k)
        hp0 = dv.highpass(x[:, 0])
        thr0 = k * dv.robust_sd(hp0)
        for o in np.round(r["stim_onsets"] * dv.FS).astype(int):
            for c in (1, 2):
                ds = dv.detect_spikes(x[:, c], k)
                for d in ds[(ds >= o) & (ds < o + dv.frames(300))]:
                    if not np.any((sp <= d) & (sp >= d - dv.frames(5))):
                        rows.append({"K": k, "rec": r["rec"], "onset_s": o / dv.FS, "dend": c, "ms_after_onset": (d - o) * 1000 / dv.FS,
                                     "nearest_soma_spike_frames": int(sp[np.argmin(np.abs(sp - d))] - d),
                                     "soma_peak_over_threshold": round(hp0[d - 3:d + 3].max() / thr0, 2),
                                     "soma_spike_within_1_frame_after": bool(np.any((sp > d) & (sp <= d + 1)))})
orph = pd.DataFrame(rows)
print(orph.to_string(index=False))

**Result: partially reproduced.** No dendrite has a delay above 3 ms (0.34 to 1.48 ms), so the
distal reading is empty and the claim is tested on all dendrites. There the first bAP of the
burst is smaller than the later ones on average (4.32 against 6.31 with K = 5; 3.92 against 6.85
with K = 4) and in 18 of 20 (K = 5) or 20 of 22 (K = 4) trial and dendrite pairs; per dendrite,
in 8 and 10 of 13 trials (K = 5) and 9 and 11 (K = 4), so dendrite 1 reaches the 9-trial bound
only with K = 4. 3 (K = 5) and 2 (K = 4) trials have fewer than two somatic spikes. Not every
detected dendritic spike follows a somatic spike: 5 of 40 (K = 5) and 8 or 9 of 76 (K = 4) have
none in the bAP window before them. None of these has a somatic spike within 1 frame after it:
the nearest somatic spike is 3 to 119 frames away, and around 13 of the 14 the soma reaches
0.54 to 1.00 of its threshold (1.64 around the one whose somatic spike is 3 frames, 6 ms,
before). So they are not frame-rounding artifacts. The soma comes near
its threshold around most of them, but the table does not show whether a somatic event below
the threshold is there.

## fig4l-sensory-plateau

> The average sensory-evoked dendritic response is a fast upstroke followed by a sustained
> plateau depolarization lasting about 400 ms, much longer than the 100 ms stimulus.

The average over the 13 trials of each dendrite's trace, triggered off puff onset (trials
shorter than the window are averaged where they have data). Readings (a) time until the average
falls back within 1 s.d. of the 500 ms pre-stimulus baseline, (b) full width at half of the
plateau (mean 50 to 150 ms). The average is smoothed over 10 ms, and "falls back" means it stays
within the band for 50 ms; both are standard choices, shown with no smoothing and 20 ms.

In [ ]:
pre, post = dv.frames(500), dv.frames(2000)
segs = {1: [], 2: []}
for r in sensory:
    x = r["data"]
    for o in np.round(r["stim_onsets"] * dv.FS).astype(int):
        for c in (1, 2):
            seg = np.full(pre + post, np.nan)
            lo, hi = o - pre, min(o + post, len(x))
            seg[:hi - lo] = x[lo:hi, c]
            segs[c].append(seg)
tl = (np.arange(pre + post) - pre) * 1000 / dv.FS
print("Trials per dendrite:", {c: len(v) for c, v in segs.items()},
      "; trials with the full 2 s after onset:", int(np.sum(~np.isnan(np.array(segs[1])[:, -1]))))


def durations(avg, smooth_ms, hold_ms=50):
    """(a) first time after 150 ms from which the average stays within 1 s.d. of the baseline
    mean for hold_ms; (b) width of the run above half the plateau that contains 50-150 ms."""
    from scipy import ndimage
    a = ndimage.uniform_filter1d(avg, max(1, dv.frames(smooth_ms))) if smooth_ms else avg.copy()
    base = a[:pre]
    mu, sd = base.mean(), base.std(ddof=1)
    post_a = a[pre:]
    inside = np.abs(post_a - mu) <= sd
    hold = dv.frames(hold_ms)
    t_a = np.nan
    for i in range(dv.frames(150), len(post_a) - hold):
        if inside[i:i + hold].all():
            t_a = i * 1000 / dv.FS
            break
    plateau = post_a[dv.frames(50):dv.frames(150)].mean() - mu
    above = (post_a - mu) >= plateau / 2
    i0 = dv.frames(100)
    start = i0
    while start > 0 and above[start - 1]:
        start -= 1
    end = i0
    while end + 1 < len(above) and above[end + 1]:
        end += 1
    return t_a, (end + 1 - start) * 1000 / dv.FS, start * 1000 / dv.FS, (end + 1) * 1000 / dv.FS, plateau, mu, sd


rows = []
avgs = {}
for c in (1, 2):
    avgs[c] = np.nanmean(np.array(segs[c]), axis=0)
avgs["mean of both"] = (avgs[1] + avgs[2]) / 2
for name, avg in avgs.items():
    for sm in (10, 0, 20):
        ta, tb, t0, t1, pl, mu, sd = durations(avg, sm)
        rows.append({"dendrite": name, "smooth_ms": sm, "a_return_ms": ta, "b_FWHM_ms": tb, "b_from_ms": t0, "b_to_ms": t1,
                     "plateau_dF": round(pl, 2), "baseline_mean": round(mu, 2), "baseline_sd": round(sd, 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("a_return_ms is NaN where the average does not return within the 2 s after onset.")
print("Expected: 400 ms, within 360 to 440; depolarization outlasting 100 ms under both readings.")
m = avgs["mean of both"]
print("Mean of both dendrites in 50 ms bins from -500 ms (Delta F):")
print(np.round(np.nanmean(m[:len(m) // 25 * 25].reshape(-1, 25), axis=1), 2).tolist())
print("Trials contributing per 50 ms bin after onset:",
      np.round(np.sum(~np.isnan(np.array(segs[1])), 0)[pre::25]).astype(int).tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, avg in avgs.items():
    ax.plot(tl, avg, lw=0.8, label=f"dendrite {name}")
ax.axvspan(0, 100, color="tab:blue", alpha=0.1)
ax.set_xlabel("time from air-puff onset (ms)")
ax.set_ylabel("Delta F")
ax.legend(fontsize=7)
plt.show()
print("Average range:", round(min(np.nanmin(v) for v in avgs.values()), 2), "to", round(max(np.nanmax(v) for v in avgs.values()), 2))

**Result: not reproduced.** The average dendritic response reaches half its 50 to 150 ms plateau
46 ms after the puff and stays above it until 310 ms, a full width at half plateau of 264 ms
(236 to 270 ms over dendrites and smoothing), outside 360 to 440 ms. It then stays above the
baseline band (mean -0.85, s.d. 0.30) for the 2 s after onset, so reading (a) has no return
time. The depolarization outlasts the 100 ms stimulus under both readings.

## figs12-spontaneous-failures-both-states

> During spontaneous activity, bAP failures occur in both anesthetized and awake recordings.

Spontaneous recordings with a soma and dendrites, split by state: the figure3i recordings
(column 1 taken as the soma) and, for the awake state, the Fig. 4a-d recordings, which the paper
records in awake mice (Results, "we first recorded spontaneous activity at the soma and apical
dendrites in awake mice"). A failure is a somatic spike whose dendritic bAP amplitude on the
high-pass filtered trace (fig4bc windows, from the 5 ms before the spike) is below 2 robust s.d.
of that trace. A variant compares with the 95th percentile of the same measure at random
spike-free times. The column 1 check of the data check bears on the anesthetized recordings.

In [ ]:
rng = np.random.default_rng(12)
rec12 = [(r["name"], r["state"], r["data"]) for r in fig3_traces] + [(f"fig4ad {r['rec']}", "awake", r["data"]) for r in spont]
rows = []
for name, state, x in rec12:
    for k in (5, 4):
        sp = dv.detect_spikes(x[:, 0], k)
        for win in (5, 8):
            n_sp, n_fail, n_fail_null = 0, 0, 0
            for c in range(1, x.shape[1]):
                hp = dv.highpass(x[:, c])
                thr = 2 * dv.robust_sd(hp)
                amps = np.array([dv.bap_amplitude(hp, s, s, win) for s in sp])
                amps = amps[np.isfinite(amps)]
                far = np.ones(len(x), bool)
                for s in sp:
                    far[max(0, s - 25):s + 25] = False
                cand = np.flatnonzero(far[10:-10]) + 10
                null = np.array([dv.bap_amplitude(hp, s, s, win) for s in rng.choice(cand, min(500, len(cand)), replace=False)])
                n_sp += len(amps)
                n_fail += int((amps < thr).sum())
                n_fail_null += int((amps < np.nanpercentile(null, 95)).sum())
            rows.append({"recording": name, "state": state, "K": k, "win_ms": win, "spikes": len(sp),
                         "spike_dend_pairs": n_sp, "failures_2sd": n_fail, "failures_null95": n_fail_null})
f12 = pd.DataFrame(rows)
summ = f12.groupby(["state", "K", "win_ms"]).agg(recordings=("recording", "nunique"),
                                                  with_spikes=("spikes", lambda s: int((s > 0).sum())),
                                                  pairs=("spike_dend_pairs", "sum"), fail_2sd=("failures_2sd", "sum"),
                                                  fail_null95=("failures_null95", "sum"))
summ["frac_2sd"] = (summ.fail_2sd / summ.pairs).round(3)
summ["frac_null95"] = (summ.fail_null95 / summ.pairs).round(3)
summ["recordings_with_failure"] = f12.assign(f=f12.failures_2sd > 0).groupby(["state", "K", "win_ms"]).f.sum()
print(summ.to_string())
print("\nPer recording, K = 5, 0-5 ms:")
print(f12[(f12.K == 5) & (f12.win_ms == 5)].to_string(index=False))
print("\nAnesthetized recordings, K = 4, 0-5 ms:")
print(f12[(f12.K == 4) & (f12.win_ms == 5) & (f12.state == "anesthetized")].to_string(index=False))
print("\nThe anesthetized failures rest on column 1 being the soma; the column 1 check in the data check finds that the"
      " data do not show it.")

**Result: partially reproduced.** Failures (dendritic bAP
below 2 s.d. of the dendrite's noise) occur in the awake recordings under every reading: 22 of
1187 spike and dendrite pairs in 7 recordings (K = 5) and 83 to 88 of 2469 in 15 (K = 4),
including 6 of the 11 Fig. 4a-d recordings at K = 5, whose column 1 is stated to be the soma. In
the anesthetized recordings 10 of 26 pairs in 3 recordings (K = 5) and 30 of 77 in 4 (K = 4)
fall below the bound, but they come from 6 (K = 5) or 17 (K = 4) events in column 1, which the
file calls the soma only "if present", and the column 1 check does not show that it is one there
(0 to 3 events at 5 s.d. per recording; in recording 13 two of five columns peak before it). The
null-based variant gives the same pattern (anesthetized 46% to 55%, awake 4% to 8%). The awake
part holds. The anesthetized part cannot be tested, as no column of those recordings is shown to
be a soma; the outcome takes no assumption about column 1. Were column 1 accepted as the soma,
the anesthetized part would pass and the claim would be reproduced.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1d-light-intensity-spike-rate | reproduced, resting on inferred pulse times |
| fig1fg-dendrite-peaks-after-soma | partially reproduced, resting on sub-frame peak timing |
| fig2c-no-trial-variation | blocked: data missing |
| fig2e-half-amplitude-distance | reproduced, resting on the LR site set |
| fig2f-delay-range | not reproduced |
| fig2f-conduction-speed | reproduced, resting on the LR site set |
| fig3d-dendrite-cross-correlation | partially reproduced, resting on the pair set |
| figs9-shot-noise-bound | blocked: data missing |
| fig3i-coupling-vs-distance | blocked: data missing |
| figs11-dendritic-spikes-rare | blocked: data missing |
| fig4a-doublet-isi | not reproduced |
| fig4bc-doublet-first-fails | partially reproduced (fails one reading at K = 5) |
| fig4d-accelerating-amplifies | partially reproduced (fails readings (a) and (c) at K = 5) |
| fig4fg-opto-first-bap-attenuated | blocked: no distal dendrite |
| fig4h-biphasic-bap-amplitude | blocked: depends on fig4fg |
| figs13-distal-more-modulated | blocked: depends on fig4h |
| fig4ijk-sensory-first-fails | partially reproduced |
| fig4l-sensory-plateau | not reproduced |
| figs12-spontaneous-failures-both-states | partially reproduced (anesthetized half not testable) |

How the analysis differs from the paper's. The dandiset holds the pipeline's per-branch
traces and summary tables, not the movies, so the background-robust dF/F, the shot-noise bound
and every per-site Fig. 2 measure come from the deposited values or are not available. The
deposited data follow a later version of the paper (more neurons per figure), and the paper's
subsets cannot be selected by identifier: Fig. 2 is tested on the LR construct's 30 sites, Fig.
3d on the Fig. 3 neuron and on the three neurons recorded in both states, Fig. 4a-d on 11
recordings from 5 mice (paper: 3 neurons). Spikes are detected here at 5 and 4 robust s.d. (the
paper gives no method), and where the criterion asks for every reading both decide; 4 s.d.
recovers more of the deposited spike times (mean recall 0.72 against 0.41). The Fig. 1 pulse
times are inferred from the traces. The distal rule of the Methods (delay above 3 ms) selects no
dendrite of the Fig. 4e-h and 4i-l recordings, while the deposited Fig. 2 delays run about 1.9 ms
below the paper's, so the delays here may not be measured as the paper's are.

What was found about the dandiset. Every file is deposited twice, byte for byte, and one
recording a third time. The files carry no light-pulse record for Fig. 1, no recordings of
dendrite-targeted stimulation (Figs. 3e-i, S11), no single-trial traces or baseline noise for
Fig. 2 (the Fig. 2h and 2i/2j files hold pooled normalized amplitudes without site labels), no
movies or pixel masks (placeholders only), and no morphology reconstructions (z-stacks only). The
Fig. 2 delay file names its delay series as amplitudes in -dF/F and repeats 17 rows three times
in another series. The Fig. 3d table holds 30 correlations above 1. The Fig. 3 neuron's
anesthetized file is described as awake. The figure3i file does not say which recordings have a
soma column, and in its anesthetized recordings column 1 does not behave as a spiking soma. The
Fig. 4a-d spike times are stated as frame indices but are seconds, one frame after the somatic
peak; their per-recording trio tables repeat one value across all columns. Two Fig. 4e-h
recordings are sampled at 1,000 Hz and the others at 500 Hz, and their stimulus lasts 259 ms
against 518 ms in the others. The Fig. 4i-l spike times are unsorted in two recordings. No
deposited delay in the Fig. 4e-h recordings exceeds 3 ms, the Methods' split for distal
dendrites, and no STA delay of a Fig. 4e-h or Fig. 4i-l dendrite does.